**Greedy information-maximization policy**

This policy selects a state with the highest predictive entropy (equivalent to mutual information maximization), moves there, and then selects the next goal.

In [ ]:
!wget -q -O box_gym.py https://raw.githubusercontent.com/MurpheyLab/boxgpt/main/box_gym.py

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation
from tqdm.auto import tqdm

from box_gym import BoxGym


def show_video(frames):
    height, width = frames[0].shape[:2]
    dpi = 100
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.axis("off")
    image = ax.imshow(frames[0])

    def update(index):
        image.set_data(frames[index])
        return (image,)

    video = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=50, blit=True
    )
    plt.close(fig)
    with plt.rc_context({"animation.embed_limit": 100.0}):
        return HTML(video.to_html5_video())

In [ ]:
def highest_uncertainty_goal(observation, rng):
    uncertainty = observation["uncertainty"]
    maxima = np.argwhere(uncertainty == uncertainty.max())
    row, column = maxima[rng.integers(len(maxima))]
    axis = np.linspace(0.0, 1.0, uncertainty.shape[0])
    return np.array([axis[column], axis[row]])


def action_toward_goal(observation, goal, env):
    direction = goal - observation["sensor_pos"]
    distance = np.linalg.norm(direction)
    if distance < 1e-12:
        return np.zeros(2, dtype=np.float32)
    speed = min(env.max_velocity, distance / env.dt)
    return (direction / distance * speed).astype(np.float32)

In [ ]:
env = BoxGym()
observation, info = env.reset(seed=12)
rng = np.random.default_rng(7)
diagnostics = True
max_steps = 300
goal = highest_uncertainty_goal(observation, rng)
frames = [env.render(diagnostics=diagnostics)]
goals_reached = 0

pbar = tqdm(range(max_steps))
for _ in pbar:
    action = action_toward_goal(observation, goal, env)
    observation, reward, done, truncated, info = env.step(action)
    frames.append(env.render(diagnostics=diagnostics))
    pbar.set_description(f"uncertainty: {info['uncertainty']:.0e}")

    reached = (
        np.linalg.norm(observation["sensor_pos"] - goal)
        <= np.finfo(np.float32).eps
    )
    if reached:
        goals_reached += 1
        goal = highest_uncertainty_goal(observation, rng)

    if done:
        break

env.close()
show_video(frames)